# Colab: Qwen3-VL-30B-A3B-Instruct 다운로드

**목적**: `WORKLOG_0922.MD` §7 Colab 세팅 계획의 1단계(모델 다운로드). 학습은 하지 않고 다운로드 + 최소 검증만 합니다.

| 항목 | 값 |
|---|---|
| HF repo id | `Qwen/Qwen3-VL-30B-A3B-Instruct` |
| 라이선스 | Apache 2.0 (상업·교육 목적 사용 허용, 확인 완료) |
| 규모 | 약 31B 파라미터, BF16, 다운로드 약 60GB |
| 로딩 클래스 | `Qwen3VLMoeForConditionalGeneration` (MoE) |

**런타임**: Colab Pro/Pro+ → 런타임 유형에서 **A100(고용량 RAM)** 으로 설정하고 시작하세요.

**진행 순서**: (1) GPU 확인 → (2) Drive 마운트 → (3) 라이브러리 설치 → (4) 모델 다운로드(Drive에 저장, 세션 끊겨도 재다운로드 불필요) → (5) 다운로드 검증(용량/파일 목록) → (6) 가벼운 로딩 확인(가중치 전체 로드 없이 processor/config만). **zero-shot 게이트(홀드아웃 채점)는 이 노트북 다음 단계**이며, 별도로 진행합니다.

## 1. GPU 확인
A100 40GB가 잡혔는지 먼저 확인합니다. 다른 GPU(T4, L4 등)로 잡히면 4bit로도 30B가 빠듯할 수 있으니 런타임을 다시 설정하세요.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 2. Google Drive 마운트
모델을 Drive에 받아두면 세션이 끊겨도(12시간 제한, 유휴 종료) 다시 받을 필요가 없습니다. 대신 학습 시 Drive(FUSE) 읽기가 로컬 디스크보다 느릴 수 있으니, 학습 단계에서는 6번 셀의 로컬 복사 옵션을 참고하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/ssafy-ai'
MODEL_DIR = f'{DRIVE_ROOT}/models/Qwen3-VL-30B-A3B-Instruct'
os.makedirs(MODEL_DIR, exist_ok=True)
print('모델 저장 위치:', MODEL_DIR)

# Drive 여유 용량 확인 (모델만 약 60GB, 여유 있게 80GB 이상 권장)
!df -h /content/drive/MyDrive 2>/dev/null || echo '(df로 Drive 용량 확인 불가 - Drive 자체 저장 공간을 웹에서 따로 확인하세요)'

## 3. 라이브러리 설치
30B-A3B(MoE)는 최신 transformers가 필요합니다. 로컬 `baseline`(4.57.6)에서도 `qwen3_vl_moe` 모듈 자체는 확인됐지만, Colab은 매번 새 환경이므로 최신판을 바로 설치합니다. 로컬과 동일하게 `peft`, `bitsandbytes`(4bit 로딩용)도 같이 설치합니다.

In [ ]:
%%capture
!pip install -U git+https://github.com/huggingface/transformers
!pip install -U accelerate peft bitsandbytes huggingface_hub pillow pandas

In [ ]:
import transformers, peft, bitsandbytes, accelerate
print('transformers', transformers.__version__)
print('peft', peft.__version__)
print('bitsandbytes', bitsandbytes.__version__)
print('accelerate', accelerate.__version__)

# 로딩 클래스가 이 버전에 존재하는지만 가볍게 확인 (모델을 실제로 불러오진 않음)
from transformers import Qwen3VLMoeForConditionalGeneration  # noqa: F401
print('Qwen3VLMoeForConditionalGeneration import 성공 - 이 transformers 버전은 30B-A3B(MoE)를 지원합니다')

## 4. 모델 다운로드
`huggingface_hub.snapshot_download`는 중간에 끊겨도 같은 셀을 다시 실행하면 이어받습니다(로컬 `download_models.py`와 같은 방식). 약 60GB라 처음엔 시간이 꽤 걸립니다.

게이트 있는(gated) 저장소가 아니라 로그인 없이 받아지는 게 보통이지만, 혹시 401/403 오류가 나면 아래 로그인 셀의 주석을 풀고 [허깅페이스 토큰](https://huggingface.co/settings/tokens)으로 로그인한 뒤 다시 실행하세요.

In [ ]:
# 필요할 때만 사용 (게이트/속도제한 등으로 401·403이 뜰 때)
# from huggingface_hub import login
# login(token='hf_...')  # https://huggingface.co/settings/tokens 에서 발급

In [ ]:
from huggingface_hub import snapshot_download

REPO_ID = 'Qwen/Qwen3-VL-30B-A3B-Instruct'

local_path = snapshot_download(
    repo_id=REPO_ID,
    local_dir=MODEL_DIR,
    local_dir_use_symlinks=False,   # Drive 위에 심볼릭 링크는 문제가 될 수 있어 실제 파일로 받음
    max_workers=4,                   # Colab 네트워크/디스크에 맞춰 조절 (느리면 2로 낮추기)
)
print('다운로드 완료:', local_path)

## 5. 다운로드 검증

In [ ]:
import os

print('=== 용량 ===')
!du -sh "$MODEL_DIR"

print('\n=== 파일 목록 ===')
for f in sorted(os.listdir(MODEL_DIR)):
    p = os.path.join(MODEL_DIR, f)
    size = os.path.getsize(p) / 1e9
    print(f'{size:8.2f} GB  {f}')

# 필수 파일 존재 확인 (없으면 다운로드가 덜 된 것 -> 4번 셀 다시 실행해서 이어받기)
must_have = ['config.json', 'tokenizer_config.json', 'preprocessor_config.json']
missing = [f for f in must_have if not os.path.exists(os.path.join(MODEL_DIR, f))]
assert not missing, f'필수 파일 누락: {missing} -> 4번 셀을 다시 실행하세요 (이어받기 지원)'
print('\n필수 파일 확인 완료')

## 6. 가벼운 로딩 확인 (전체 가중치는 아직 안 불러옴)
config/tokenizer/processor만 불러와서 손상 없이 받아졌는지 빠르게 확인합니다. 전체 모델 로딩(4bit)은 다음 단계인 zero-shot 게이트에서 진행합니다.

In [ ]:
from transformers import AutoConfig, AutoProcessor

cfg = AutoConfig.from_pretrained(MODEL_DIR, local_files_only=True)
proc = AutoProcessor.from_pretrained(MODEL_DIR, local_files_only=True)
print('config 로딩 성공. model_type =', cfg.model_type)
print('processor 로딩 성공')

## (선택) 7. 학습 시 빠른 I/O를 위해 로컬 디스크로 복사
Drive(FUSE)는 랜덤 읽기가 느려서, 실제 학습 단계에서는 로컬 `/content`로 복사해두면 더 빠를 수 있습니다. **세션이 끊기면 `/content`는 사라지므로 원본은 항상 Drive(`MODEL_DIR`)에 유지**하세요. 로컬 디스크 여유 공간(약 65GB 이상)이 있는지 먼저 확인합니다.

In [ ]:
!df -h /content

# 여유 공간이 충분하면 아래 주석을 풀고 실행
# LOCAL_MODEL_DIR = '/content/models/Qwen3-VL-30B-A3B-Instruct'
# !mkdir -p "$LOCAL_MODEL_DIR"
# !rsync -ah --info=progress2 "$MODEL_DIR"/ "$LOCAL_MODEL_DIR"/

## 다음 단계
다운로드가 끝났으면, 팀 저장소의 `src/score.py`(또는 그대로 이식한 스크립트)로 **zero-shot 게이트**를 돌립니다.

```bash
python src/score.py --split holdout --model "$MODEL_DIR" --max-tokens 768 --quant 4bit --tag q3vl30b_gate
```

홀드아웃 1,000장 기준, 로컬 최고 앙상블(95.6%)·최고 단일(94.7%) 대비 압축 손실을 감안하고도 확실히 나은지 확인한 뒤 학습 여부를 결정합니다 (`WORKLOG_0922.MD` §7.3).